In [1]:
import pandas as pd
import numpy as np
import json
from utils.tv_var.harmonic_standardizer import HarmonicStandardizer
from utils.tv_var.periodic_var import PeriodicVAR, select_order

import matplotlib.pyplot as plt

pcs_sel_6h = pd.read_csv('outputs/pca/pcs_msl_30_6h.csv', index_col=0, parse_dates=True)

#### Quitar media y desviación estacionales (la estacionalidad va aquí, no en los coeficientes)
harm_std_pcs = HarmonicStandardizer(n_harmonics=3)
harm_std_pcs.fit(pcs_sel_6h)
pcs_sel_6h_z = harm_std_pcs.transform(pcs_sel_6h)
print(pcs_sel_6h_z.shape)

(126864, 30)


In [2]:
bic_0b = select_order(pcs_sel_6h_z, p_values=[6, 8, 10, 12, 16], n_harm=0, freq="6h")
bic_0b

,nobs,loglike,n_params,bic,best
p,,,,,
6,126848,133736.734172,11010,-138097.768115,False
8,126848,163350.344773,12810,-176173.648681,False
10,126848,188280.959276,14610,-204883.537050,True
12,126848,195264.530949,16410,-197699.339759,False
16,126848,212314.258515,20010,-189496.113618,False


In [3]:
var_h0 = PeriodicVAR(p=10, n_harm=0, freq="6h").fit(pcs_sel_6h_z)
var_h1 = PeriodicVAR(p=10, n_harm=1, freq="6h").fit(pcs_sel_6h_z)
var_h2 = PeriodicVAR(p=10, n_harm=2, freq="6h").fit(pcs_sel_6h_z)

pd.DataFrame({"n_harm=0": [var_h0.n_params(), var_h0.loglike(), var_h0.bic()],
              "n_harm=1": [var_h1.n_params(), var_h1.loglike(), var_h1.bic()],
              "n_harm=2": [var_h2.n_params(), var_h2.loglike(), var_h2.bic()]},
             index=["n_params", "loglike", "bic"])

,n_harm=0,n_harm=1,n_harm=2
n_params,14610.000000,32670.000000,50730.000000
loglike,188255.448277,331671.406146,365560.029048
bic,-204831.824004,-279444.434456,-135002.374972


In [4]:
for n, v in {"h0": var_h0, "h1": var_h1, "h2": var_h2}.items():
    print(n, v.stability())

h0 {'rho': 0.3771946247326547, 'rho_daily': 0.9973323481437106, 'rho_step': 0.9993324188364805, 'stable': True}
h1 {'rho': 0.37216114138159284, 'rho_daily': 0.9972956405294687, 'rho_step': 0.9993232234024519, 'stable': True}
h2 {'rho': 0.3888158460620039, 'rho_daily': 0.9974152655060792, 'rho_step': 0.9993531891005693, 'stable': True}


In [5]:
print(var_h2.summary())

                         Periodic VAR                         
Variables (k)                   30    msl_PC1, msl_PC2, msl_PC3, msl_PC4, msl_PC5, msl_PC6, msl_PC7, msl_PC8, msl_PC9, msl_PC10, msl_PC11, msl_PC12, msl_PC13, msl_PC14, msl_PC15, msl_PC16, msl_PC17, msl_PC18, msl_PC19, msl_PC20, msl_PC21, msl_PC22, msl_PC23, msl_PC24, msl_PC25, msl_PC26, msl_PC27, msl_PC28, msl_PC29, msl_PC30
Frequency                       6h    0 days 06:00:00
Exogenous (q)                    0    -
Lags (p)                        10
Harmonics (n_harm)               2
Noise                       gaussian / monthly
Sample                    1940-01-03 12:00 → 2026-10-31 18:00
Observations (nobs)         126854
--------------------------------------------------------------
Coefficients                 45150    (1505 per equation)
Covariance parameters         5580    (12 block(s))
Total parameters             50730
Log-likelihood                 365560.03
BIC                           -135002.37
-----------

#### Simulation

In [6]:
import os, pickle, time

out = "outputs/var_6h"
os.makedirs(out, exist_ok=True)

#### Estandarizador (común a los tres) y los tres modelos, cada uno con su sufijo
pickle.dump(harm_std_pcs, open(f"{out}/harm_std_pcs.pkl", "wb"))
modelos = {0: var_h0, 1: var_h1, 2: var_h2}
for n, v in modelos.items():
    pickle.dump(v, open(f"{out}/var_h{n}.pkl", "wb"))

#### Simulaciones: 100 para producción (h1), 20 para comparar (h0, h2)
burn = int(pd.Timedelta(days=3*365) / pd.Timedelta("6h"))
n_sims = {0: 20, 1: 100, 2: 20}

for n, v in modelos.items():
    sim_dir = f"{out}/simulaciones_h{n}"
    os.makedirs(sim_dir, exist_ok=True)
    for i in range(n_sims[n]):
        t = time.time()
        sim_z = v.simulate(n_years=87, start="1940-01-01", seed=i, burn=burn)
        harm_std_pcs.inverse_transform(sim_z).astype("float32").to_parquet(f"{sim_dir}/sim_{i:03d}.parquet")
        print(f"h{n} sim {i:03d}  ({time.time() - t:.0f} s)")

h0 sim 000  (2 s)
h0 sim 001  (2 s)
h0 sim 002  (2 s)
h0 sim 003  (2 s)
h0 sim 004  (2 s)
h0 sim 005  (2 s)
h0 sim 006  (2 s)
h0 sim 007  (2 s)
h0 sim 008  (2 s)
h0 sim 009  (2 s)
h0 sim 010  (2 s)
h0 sim 011  (2 s)
h0 sim 012  (2 s)
h0 sim 013  (2 s)
h0 sim 014  (2 s)
h0 sim 015  (2 s)
h0 sim 016  (2 s)
h0 sim 017  (2 s)
h0 sim 018  (2 s)
h0 sim 019  (3 s)
h1 sim 000  (2 s)
h1 sim 001  (2 s)
h1 sim 002  (2 s)
h1 sim 003  (2 s)
h1 sim 004  (2 s)
h1 sim 005  (2 s)
h1 sim 006  (2 s)
h1 sim 007  (3 s)
h1 sim 008  (2 s)
h1 sim 009  (2 s)
h1 sim 010  (2 s)
h1 sim 011  (2 s)
h1 sim 012  (2 s)
h1 sim 013  (2 s)
h1 sim 014  (2 s)
h1 sim 015  (2 s)
h1 sim 016  (2 s)
h1 sim 017  (2 s)
h1 sim 018  (2 s)
h1 sim 019  (2 s)
h1 sim 020  (2 s)
h1 sim 021  (2 s)
h1 sim 022  (2 s)
h1 sim 023  (2 s)
h1 sim 024  (2 s)
h1 sim 025  (2 s)
h1 sim 026  (2 s)
h1 sim 027  (2 s)
h1 sim 028  (2 s)
h1 sim 029  (2 s)
h1 sim 030  (2 s)
h1 sim 031  (2 s)
h1 sim 032  (2 s)
h1 sim 033  (2 s)
h1 sim 034  (2 s)
h1 sim 035